# Flow Matching (using flow_matching package)

In [1]:
import time
import torch
from torch import nn, Tensor
from torch.utils.data import TensorDataset, DataLoader, random_split

# flow_matching
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper

# visualization
import matplotlib.pyplot as plt

In [17]:
class Flow(nn.Module):
    def __init__(self, dim: int = 32, h: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h), nn.ELU(),
            nn.Linear(h, h), nn.ELU(),
            nn.Linear(h, h), nn.ELU(),
            nn.Linear(h, dim))
    
    def forward(self, t: Tensor, x_t: Tensor) -> Tensor:
        return self.net(torch.cat((t, x_t), -1))
    
    def step(self, x_t: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor:
        t_start = t_start.view(1, 1).expand(x_t.shape[0], 1)
        
        return x_t + (t_end - t_start) * self(t=t_start + (t_end - t_start) / 2, x_t= x_t + self(x_t=x_t, t=t_start) * (t_end - t_start) / 2)

In [18]:
num_epochs = 50
batch_size = 4

attenu_latent = torch.load('attenu_latent.pt')
attenu_latent.shape

attenu_latent_dataset = TensorDataset(attenu_latent, attenu_latent)

train_size = (int(0.7 * len(attenu_latent)) // 4) * 4
val_size = int((len(attenu_latent) - train_size)/2)
test_size = int(len(attenu_latent) - train_size - val_size)

generator = torch.Generator().manual_seed(42)
attenu_latent_train_dataset, attenu_latent_val_dataset, attenu_latent_test_dataset = random_split(attenu_latent_dataset, [train_size, val_size, test_size], generator=generator)

attenu_latent_train_dataloader = DataLoader(attenu_latent_train_dataset, batch_size = batch_size, shuffle = 1)
attenu_latent_val_dataloader = DataLoader(attenu_latent_val_dataset, batch_size = batch_size, shuffle = 1)
attenu_latent_test_dataloader = DataLoader(attenu_latent_test_dataset, batch_size = batch_size, shuffle = 1)

In [19]:
num_epochs = 50

flow = Flow()

optimizer = torch.optim.Adam(flow.parameters(), 1e-2)
loss_fn = nn.MSELoss()

epoch_loss = []
epoch_loss_val = []

for epoch in range(num_epochs):
    start = torch.Event(enable_timing=True)
    end = torch.Event(enable_timing=True)

    start.record()

    flow.train()
    running_loss = 0.0
    running_loss_val = 0.0
    for train_batch, _ in attenu_latent_train_dataloader:
        x_1 = train_batch
        x_0 = torch.randn_like(x_1)
        t = torch.rand(len(x_1), 1)
        
        x_t = (1 - t) * x_0 + t * x_1
        dx_t = x_1 - x_0
        
        optimizer.zero_grad()
        loss = loss_fn(flow(t=t, x_t=x_t), dx_t)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * train_batch.size(0)

    for val_batch, _ in attenu_latent_val_dataloader:
        x_1 = val_batch
        x_0 = torch.randn_like(x_1)
        t = torch.rand(len(x_1), 1)
        
        x_t = (1 - t) * x_0 + t * x_1
        dx_t = x_1 - x_0

        loss = loss_fn(flow(t=t, x_t=x_t), dx_t)

        running_loss_val += loss.item() * val_batch.size(0)
    
    epoch_loss.append(running_loss / len(attenu_latent_train_dataloader.dataset))
    epoch_loss_val.append(running_loss_val / len(attenu_latent_val_dataloader.dataset))

    end.record()

    print(f"Epoch [{epoch + 1}/{num_epochs}], loss: {epoch_loss[epoch]:.4f}, validation loss: {epoch_loss_val[epoch]:.4f}, time elapsed = {start.elapsed_time(end)/1000:.2f} s")
    
    # if epoch > 1:
    #     if epoch_loss_val[-1] > epoch_loss_val[-2]:
    #         print("Validation loss increased, early stopping")
    #         break

    

Epoch [1/50], loss: 1.0432, validation loss: 0.8461, time elapsed = 0.03 s
Epoch [2/50], loss: 0.8482, validation loss: 0.7782, time elapsed = 0.02 s
Epoch [3/50], loss: 0.7804, validation loss: 0.6557, time elapsed = 0.01 s
Epoch [4/50], loss: 0.7271, validation loss: 0.6617, time elapsed = 0.01 s
Epoch [5/50], loss: 0.6635, validation loss: 0.6413, time elapsed = 0.01 s
Epoch [6/50], loss: 0.6666, validation loss: 0.6844, time elapsed = 0.01 s
Epoch [7/50], loss: 0.6890, validation loss: 0.6417, time elapsed = 0.01 s
Epoch [8/50], loss: 0.6467, validation loss: 0.6181, time elapsed = 0.01 s
Epoch [9/50], loss: 0.5740, validation loss: 0.5961, time elapsed = 0.01 s
Epoch [10/50], loss: 0.6043, validation loss: 0.6935, time elapsed = 0.01 s
Epoch [11/50], loss: 0.5358, validation loss: 0.5830, time elapsed = 0.01 s
Epoch [12/50], loss: 0.5486, validation loss: 0.5106, time elapsed = 0.01 s
Epoch [13/50], loss: 0.5630, validation loss: 0.6030, time elapsed = 0.01 s
Epoch [14/50], loss: 